# 🌽 CornSight — Part 2: From Corn Maps to Market Intelligence

**Notebook 1** (`cornsight_train.ipynb`) trained the Prithvi satellite model and produced `chip_results.csv`: for 771 satellite images across the US, how much corn is planted and how healthy it looks.

**This notebook** turns that into business intelligence:

| Part | What happens |
|---|---|
| **A** | Work out which **state** each satellite image is in, and summarize corn acreage + corn health per state |
| **B** | Get official **USDA** corn yields per state (2017–2022) and find which states had a bad 2022 |
| **C** | Get **corn futures prices** for 2022 and measure how the market moved on USDA report days |
| **D** | Store everything in **MongoDB** (images, states, USDA data, prices, reports, charts) |
| **E** | **Analysis + charts**: did the satellite see what USDA later confirmed? What is that worth? |
| **F** | **AI agent**: an open LLM (Qwen3-4B) that queries MongoDB with tools, reasons over the data, and writes market briefs for a trading desk and an ag lender. Briefs are saved back to MongoDB |

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU` (only Part F needs it).
2. The 🔑 **Secrets** panel (left sidebar) must contain `MONGODB_URI` and `NASS_API_KEY`, with **Notebook access** switched on for both.
3. Nothing else: Notebook 1's results (`chip_results.csv`, scores, charts) are read straight from the public GitHub repo [`AlonsoManchego/cornsight`](https://github.com/AlonsoManchego/cornsight/tree/main/training_outputs).

⏱️ ~15–20 min total (Part F downloads an 8 GB model the first time).

## Setup — install, load secrets, connect to MongoDB

In [ ]:
!pip install -q pymongo yfinance reverse_geocoder 'transformers>=4.56'

In [ ]:
import os, json, re, time, io, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

try:                                   # Google Colab
    from google.colab import userdata
    MONGODB_URI = userdata.get("MONGODB_URI")
    NASS_API_KEY = userdata.get("NASS_API_KEY")
except ImportError:                    # running locally: read secrets from a .env file
    from dotenv import load_dotenv
    load_dotenv(".env")
    MONGODB_URI = os.environ["MONGODB_URI"]
    NASS_API_KEY = os.environ["NASS_API_KEY"]

# Notebook 1's results live in the public GitHub repo
GITHUB_REPO = "AlonsoManchego/cornsight"
TRAINING_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/training_outputs"

# This notebook's own outputs (charts, brief files) go to a local folder; the data itself goes to MongoDB
OUT_DIR = "/content/cornsight_outputs" if os.path.exists("/content") else "./cornsight_outputs"
CHART_DIR = os.path.join(OUT_DIR, "pipeline_charts")
os.makedirs(CHART_DIR, exist_ok=True)
print("Training results from:", TRAINING_URL)
print("chip_results.csv reachable:", requests.head(f"{TRAINING_URL}/chip_results.csv").status_code == 200)
print("Secrets loaded:", bool(MONGODB_URI), bool(NASS_API_KEY))

In [ ]:
from pymongo import MongoClient, ASCENDING
import gridfs

client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=20000)
client.admin.command("ping")
db = client["cornsight"]
print("✅ Connected to MongoDB Atlas — database 'cornsight'")
print("Existing collections:", db.list_collection_names())

def save_collection(name, records, key=None):
    """Replace a MongoDB collection with fresh records (safe to re-run)."""
    db[name].delete_many({})
    if records:
        db[name].insert_many(records)
    if key:
        db[name].create_index([(key, ASCENDING)])
    print(f"  💾 {name}: {db[name].count_documents({})} documents")

def clean(df):
    """Pandas → plain Python dicts MongoDB can store (NaN → None, numpy → float/int)."""
    return json.loads(df.to_json(orient="records"))

---
## Part A — Satellite results by state
Each satellite image ("chip") has a latitude/longitude. We look up its US state offline, then summarize per state:
- **Corn share**: % of land the model sees as corn (→ planted acreage)
- **Corn health (peak season)**: average NDVI of corn pixels in photos taken in **July–August**, when corn is greenest. NDVI runs from 0 (bare soil) to ~0.9 (dense, healthy crop). Photos from March or September would be misleading (no crop yet / harvest), so we only use peak-season photos.

In [ ]:
import reverse_geocoder as rg

chips = pd.read_csv(f"{TRAINING_URL}/chip_results.csv")          # Notebook 1's output, from GitHub
geo = rg.search(list(zip(chips.lat, chips.lon)), mode=1)
chips["country"] = [g["cc"] for g in geo]
chips["state"] = [g["admin1"].upper() for g in geo]
chips = chips[chips.country == "US"].copy()

def peak_ndvi(row):
    """Highest corn NDVI among photos taken in July or August (None if no such photo / no corn)."""
    vals = [row[f"corn_ndvi_{t}"] for t in (1, 2, 3)
            if pd.to_datetime(row[f"date_{t}"]).month in (7, 8) and pd.notna(row[f"corn_ndvi_{t}"])]
    return max(vals) if vals else np.nan

chips["corn_health_peak"] = chips.apply(peak_ndvi, axis=1)
print(f"{len(chips)} US chips in {chips.state.nunique()} states; "
      f"{chips.corn_health_peak.notna().sum()} have a peak-season photo with corn")

states = chips.groupby("state").agg(
    chips=("chip_id", "count"),
    corn_share_pred=("corn_share_pred", "mean"),
    corn_share_true=("corn_share_true", "mean"),
    corn_health_peak=("corn_health_peak", "mean"),
    peak_photos=("corn_health_peak", "count"),
).reset_index()
states.sort_values("corn_share_pred", ascending=False).head(10).round(3)

---
## Part B — Official USDA corn yields (2017–2022)
From the USDA **NASS Quick Stats API**. Yield = bushels of corn per acre. We compare each state's 2022 yield with its own 2017–2021 average to see who had a bad year (2022 had a drought in the western Corn Belt).

The USDA API can be very slow, so the data is fetched **once** and stored in MongoDB; later runs read it from MongoDB. Set `REFRESH_USDA = True` to force a fresh download.

In [ ]:
REFRESH_USDA = False
NASS_URL = "https://quickstats.nass.usda.gov/api/api_GET/"
USDA_SERIES = {
    "yield_bu_acre":   "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",
    "acres_harvested": "CORN, GRAIN - ACRES HARVESTED",
    "production_bu":   "CORN, GRAIN - PRODUCTION, MEASURED IN BU",
}

def fetch_nass(short_desc, tries=3):
    params = {"key": NASS_API_KEY, "short_desc": short_desc, "agg_level_desc": "STATE",
              "source_desc": "SURVEY", "reference_period_desc": "YEAR",
              "year__GE": 2017, "year__LE": 2022, "format": "JSON"}
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(NASS_URL, params=params, timeout=300)
            r.raise_for_status()
            return pd.DataFrame(r.json()["data"])
        except Exception as e:
            print(f"   attempt {attempt} failed ({type(e).__name__}); retrying..." if attempt < tries else f"   giving up: {e}")
            time.sleep(5)
    raise RuntimeError("USDA API not responding — try again in a few minutes")

if REFRESH_USDA or db.usda_yields.count_documents({}) == 0:
    print("Downloading from the USDA API (can take 1–3 min)...")
    frames = []
    for col, desc in USDA_SERIES.items():
        d = fetch_nass(desc)
        d = d[d.prodn_practice_desc == "ALL PRODUCTION PRACTICES"]
        d["value"] = pd.to_numeric(d.Value.astype(str).str.replace(",", ""), errors="coerce")
        frames.append(d[["state_name", "year", "value"]].assign(metric=col))
    usda_long = pd.concat(frames)
    usda_long["year"] = usda_long.year.astype(int)
    save_collection("usda_yields", clean(usda_long), key="state_name")
else:
    print("Reading USDA data stored in MongoDB")
usda_long = pd.DataFrame(list(db.usda_yields.find({}, {"_id": 0})))

usda = usda_long.pivot_table(index="state_name", columns=["metric", "year"], values="value")
usda_state = pd.DataFrame({
    "usda_yield_2022": usda[("yield_bu_acre", 2022)],
    "usda_yield_avg_17_21": usda["yield_bu_acre"][[2017, 2018, 2019, 2020, 2021]].mean(axis=1),
    "production_2022_bu": usda[("production_bu", 2022)],
    "production_avg_17_21_bu": usda["production_bu"][[2017, 2018, 2019, 2020, 2021]].mean(axis=1),
    "acres_harvested_2022": usda[("acres_harvested", 2022)],
})
usda_state["yield_change_pct"] = (usda_state.usda_yield_2022 / usda_state.usda_yield_avg_17_21 - 1) * 100
usda_state = usda_state.reset_index().rename(columns={"state_name": "state"})
print(f"USDA yields for {len(usda_state)} states")
usda_state.sort_values("yield_change_pct").head(8).round(1)

---
## Part C — Corn futures and USDA report days
Corn futures (`ZC=F`, Chicago, priced in **cents per bushel**; one contract = 5,000 bushels). Every month the USDA publishes the **WASDE** report (World Agricultural Supply and Demand Estimates) with its crop forecast; traders watch it closely because a surprise moves prices instantly.

We measure how much corn moved on WASDE days compared with normal days.

In [ ]:
import yfinance as yf

# 2022 WASDE release dates. 'verified' = confirmed against a USDA source; others follow USDA's standard monthly calendar.
WASDE_2022 = [("2022-01-12", False), ("2022-02-09", False), ("2022-03-09", False), ("2022-04-08", False),
              ("2022-05-12", True),  ("2022-06-10", False), ("2022-07-12", False), ("2022-08-12", True),
              ("2022-09-12", False), ("2022-10-12", True),  ("2022-11-09", False), ("2022-12-09", False)]

if db.corn_prices.count_documents({}) == 0:
    raw = yf.download("ZC=F", start="2021-12-01", end="2023-01-31", progress=False, auto_adjust=False)
    raw.columns = raw.columns.get_level_values(0)
    raw = raw.reset_index()[["Date", "Open", "High", "Low", "Close"]]
    raw["Date"] = raw.Date.dt.strftime("%Y-%m-%d")
    save_collection("corn_prices", clean(raw), key="Date")
prices = pd.DataFrame(list(db.corn_prices.find({}, {"_id": 0}))).sort_values("Date")
prices["Date"] = pd.to_datetime(prices.Date)
prices = prices.set_index("Date")
prices["ret_pct"] = prices.Close.pct_change() * 100
# ZC=F is a "continuous" series that switches to the next contract when one expires. On 2022-07-15 it switched from the
# July (old-crop) to the September (new-crop) contract, creating a fake -13% "move". We exclude that day from returns.
ROLL_DAYS = ["2022-07-15"]
prices.loc[prices.index.isin(pd.to_datetime(ROLL_DAYS)), "ret_pct"] = np.nan
p22 = prices.loc["2022"]

wasde = []
for d, verified in WASDE_2022:
    day = pd.Timestamp(d)
    if day in p22.index:
        wasde.append({"date": d, "verified_date": verified,
                      "close_cents": float(p22.loc[day, "Close"]),
                      "move_pct": round(float(p22.loc[day, "ret_pct"]), 2)})
save_collection("wasde_reports", wasde, key="date")
wasde_df = pd.DataFrame(wasde)

normal = p22.ret_pct.drop(pd.to_datetime(wasde_df.date), errors="ignore").abs().mean()
report = wasde_df.move_pct.abs().mean()
print(f"Average daily move on WASDE days:   {report:.2f}%")
print(f"Average daily move on normal days:  {normal:.2f}%")
print(f"→ Report days move corn {report/normal:.1f}× more than a normal day on average")
big = wasde_df.loc[wasde_df.move_pct.abs().idxmax()]
print(f"Biggest report-day move: {big.date} → {big.move_pct:+.2f}% in one day")
wasde_df

---
## Part D — Store everything in MongoDB
Collections in the `cornsight` database:

| Collection | One document = | Used by |
|---|---|---|
| `chips` | one satellite image: location, state, dates, corn share, corn health | charts, agent |
| `states` | one state: satellite summary + USDA yields | charts, agent |
| `usda_yields` | one USDA number (state, year, metric) | Part B |
| `corn_prices` | one trading day of corn futures | Part C, agent |
| `wasde_reports` | one USDA report day and the price move | agent |
| `model_metrics` | the satellite model's accuracy scores | agent |
| `images` (GridFS) | every chart + corn map as a PNG file | presentation |
| `briefs` | each market brief the AI agent writes | Part F |

In [ ]:
# Combine satellite + USDA per state
state_table = states.merge(usda_state, on="state", how="left")
major = state_table[(state_table.peak_photos >= 3) & state_table.usda_yield_2022.notna()]
# Flags only for MAJOR corn states: ≥ 300 million bushels/year on average in 2017–21 (~90% of US corn; uses only
# information available before 2022), with ≥ 3 peak-season satellite photos. Compared with each other.
corn_states = major[major.production_avg_17_21_bu >= 300e6]
lo, hi = corn_states.corn_health_peak.quantile([0.33, 0.67])
state_table["health_rank"] = state_table.corn_health_peak.rank(ascending=False, method="min")
is_corn_state = state_table.state.isin(corn_states.state)
state_table["satellite_flag"] = np.select(
    [~is_corn_state, state_table.corn_health_peak < lo, state_table.corn_health_peak > hi],
    ["not a major corn state / too few photos", "STRESSED (bottom third of corn states)", "HEALTHY (top third of corn states)"],
    default="normal")
print("Satellite flags among corn states:")
print(state_table[is_corn_state].sort_values("corn_health_peak")[["state", "corn_health_peak", "satellite_flag"]].to_string(index=False))

scores = pd.read_csv(f"{TRAINING_URL}/test_scores.csv", index_col=0).iloc[:, 0]
metrics = {k.replace("test/", ""): (float(v) if k.startswith("test/") else v) for k, v in scores.items()
           if k.startswith("test/IoU_") or k in ("test/mIoU", "test/Pixel_Accuracy", "backbone", "epochs", "train_minutes")}
metrics["acreage_correlation"] = round(float(chips[["corn_share_true", "corn_share_pred"]].corr().iloc[0, 1]), 3)
metrics["reference_ibm_corn_iou"] = 0.5491

save_collection("chips", clean(chips.drop(columns=["country"])), key="state")
save_collection("states", clean(state_table), key="state")
save_collection("model_metrics", [metrics])
print("\nCollections now in MongoDB:", sorted(db.list_collection_names()))

In [ ]:
# Quick demo: query MongoDB directly (what the agent's tools do under the hood)
print("Top 5 corn states by satellite-measured corn share (queried from MongoDB):")
for s in db.states.find({"peak_photos": {"$gte": 3}}, {"_id": 0, "state": 1, "corn_share_pred": 1, "corn_health_peak": 1}) \
                  .sort("corn_share_pred", -1).limit(5):
    print(f"  {s['state']:14s} corn {s['corn_share_pred']*100:4.1f}% of land | health {s['corn_health_peak']:.2f}")

---
## Part E — Analysis: did the satellite see it coming?
**Timing matters for the business case:**
- Satellite photos from **July–August 2022** were available during the growing season.
- USDA's **final** 2022 yields came out in **January 2023**.

So if peak-season corn health lines up with final yields, the satellite gave an early warning months before the official numbers.

In [ ]:
plot = major.copy()
r_yield = plot.corn_health_peak.corr(plot.usda_yield_2022)
r_change = plot.corn_health_peak.corr(plot.yield_change_pct)

fig, ax = plt.subplots(figsize=(10, 6.5))
sc = ax.scatter(plot.corn_health_peak, plot.usda_yield_2022, s=plot.chips * 8 + 20, c=plot.yield_change_pct,
                cmap="RdYlGn", vmin=-15, vmax=15, edgecolor="k", linewidth=0.5)
for _, r in plot.iterrows():
    ax.annotate(r.state.title(), (r.corn_health_peak, r.usda_yield_2022), fontsize=8,
                xytext=(5, 3), textcoords="offset points")
m, b = np.polyfit(plot.corn_health_peak, plot.usda_yield_2022, 1)
xs = np.linspace(plot.corn_health_peak.min(), plot.corn_health_peak.max(), 50)
ax.plot(xs, m * xs + b, "k--", lw=1)
plt.colorbar(sc, label="2022 yield vs. 2017–21 average (%)")
ax.set_xlabel("Satellite corn health, Jul–Aug 2022 (NDVI)")
ax.set_ylabel("USDA final 2022 corn yield (bushels/acre)")
ax.set_title(f"Satellite corn health vs. official yield, by state (correlation = {r_yield:.2f})\nBubble size = number of satellite images")
plt.tight_layout(); plt.savefig(os.path.join(CHART_DIR, "health_vs_yield.png"), dpi=150); plt.show()
print(f"Correlation with 2022 yield: {r_yield:.2f} | with yield change vs. 5-yr average: {r_change:.2f} | states: {len(plot)}")

In [ ]:
# Major corn states only (same 12 states the flags use)
belt = state_table[is_corn_state].sort_values("corn_health_peak")
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#c0392b" if v < 0 else "#27ae60" for v in belt.yield_change_pct]
ax.barh(belt.state.str.title(), belt.yield_change_pct, color=colors)
xmax = belt.yield_change_pct.abs().max() * 1.15
for i, h in enumerate(belt.corn_health_peak):
    ax.text(xmax * 1.02, i, f"{h:.2f}", va="center", fontsize=9)
ax.text(xmax * 1.02, len(belt) - 0.4, "health", fontsize=8, color="gray")
ax.set_xlim(-xmax, xmax * 1.18)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("2022 yield vs. 2017–21 average (%)")
ax.set_title("Major corn states, sorted by satellite corn health (least healthy at the bottom)")
plt.tight_layout(); plt.savefig(os.path.join(CHART_DIR, "corn_belt_yield_change.png"), dpi=150); plt.show()

In [ ]:
# Scorecard: did the satellite's in-season flags match what USDA confirmed months later?
sc = state_table[is_corn_state & (state_table.satellite_flag != "normal")].copy()
sc["usda_result"] = np.where(sc.yield_change_pct < 0, "below average", "above average")
sc["satellite_right"] = np.where(sc.satellite_flag.str.startswith("STRESSED"), sc.yield_change_pct < 0, sc.yield_change_pct >= 0)
print(sc.sort_values("corn_health_peak")[["state", "corn_health_peak", "satellite_flag", "yield_change_pct", "usda_result", "satellite_right"]]
        .round(3).to_string(index=False))
hits, n = int(sc.satellite_right.sum()), len(sc)
print(f"\n🎯 Satellite flags matched USDA's final result in {hits} of {n} major corn states")
db.model_metrics.update_one({}, {"$set": {"flag_scorecard": {"correct": hits, "total": n,
    "details": clean(sc[["state", "satellite_flag", "yield_change_pct", "satellite_right"]])}}})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(p22.index, p22.Close / 100, color="#b98300", lw=1.8)
for _, w in wasde_df.iterrows():
    d = pd.Timestamp(w.date)
    ax.axvline(d, color="gray", ls=":", lw=0.9)
    ax.annotate(f"{w.move_pct:+.1f}%", (d, w.close_cents / 100), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8, color="#c0392b" if w.move_pct < 0 else "#27ae60")
roll = pd.Timestamp(ROLL_DAYS[0])
ax.annotate("contract switch\n(July → Sept), not a real move", (roll, p22.loc[roll, "Close"] / 100), xytext=(60, -40),
            textcoords="offset points", fontsize=8, color="gray", arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_ylabel("Corn futures ($ per bushel)")
ax.set_title("Corn futures in 2022 — dotted lines = USDA WASDE report days (with that day's move)")
plt.tight_layout(); plt.savefig(os.path.join(CHART_DIR, "corn_futures_wasde.png"), dpi=150); plt.show()

### What is an early warning worth?
Simple, transparent numbers built from the data above.

In [ ]:
avg_price = p22.Close.mean() / 100                                    # $ per bushel
prod_2022 = usda_state.production_2022_bu.sum()                       # bushels (sum of surveyed states)
crop_value = prod_2022 * avg_price
contract_value = 5000 * avg_price
report_move_dollars = contract_value * report / 100
stressed = state_table[state_table.satellite_flag.str.startswith("STRESSED")]
stressed_prod = stressed.production_2022_bu.sum()

value = {
    "avg_corn_price_2022_usd_bu": round(avg_price, 2),
    "us_corn_production_2022_bu": int(prod_2022),
    "us_corn_crop_value_2022_usd": int(crop_value),
    "one_contract_value_usd": round(contract_value),
    "avg_wasde_day_move_pct": round(report, 2),
    "avg_normal_day_move_pct": round(normal, 2),
    "avg_wasde_day_move_per_contract_usd": round(report_move_dollars),
    "biggest_wasde_day": big.date, "biggest_wasde_day_move_pct": float(big.move_pct),
    "biggest_wasde_day_move_per_contract_usd": round(contract_value * abs(big.move_pct) / 100),
    "stressed_states": stressed.state.str.title().tolist(),
    "stressed_states_share_of_production_pct": round(100 * stressed_prod / prod_2022, 1),
}
db.model_metrics.update_one({}, {"$set": {"business_value": value}})

print(f"US corn crop 2022:             {prod_2022/1e9:.1f} billion bushels  ×  ${avg_price:.2f}  ≈  ${crop_value/1e9:.0f} billion")
print(f"One futures contract:          5,000 bu × ${avg_price:.2f} = ${contract_value:,.0f}")
print(f"Average WASDE-day move:        {report:.2f}% → ${report_move_dollars:,.0f} per contract in one day")
print(f"Biggest WASDE-day move:        {big.date}: {big.move_pct:+.2f}% → ${value['biggest_wasde_day_move_per_contract_usd']:,} per contract")
print(f"Satellite-flagged states:      {', '.join(value['stressed_states'])}")
print(f"  → together {value['stressed_states_share_of_production_pct']}% of US corn production")

In [ ]:
# Upload every chart + corn map into MongoDB GridFS (so the database holds the visual results too)
fs = gridfs.GridFS(db, collection="images")
def store_png(name, data):
    for old in fs.find({"filename": name}):
        fs.delete(old._id)
    fs.put(data, filename=name, content_type="image/png", kind="corn_map" if name.startswith("map_") else "chart")

# Notebook 1's charts and corn maps, listed and downloaded from GitHub
listing = requests.get(f"https://api.github.com/repos/{GITHUB_REPO}/contents/training_outputs").json()
for item in listing:
    if item["name"].endswith(".png"):
        store_png(item["name"], requests.get(item["download_url"]).content)
# This notebook's own charts
for path in glob.glob(os.path.join(CHART_DIR, "*.png")):
    with open(path, "rb") as f:
        store_png(os.path.basename(path), f.read())
print(f"💾 images (GridFS): {db['images.files'].count_documents({})} PNG files stored")

---
## Part F — The CornSight AI agent 🤖
An open-source LLM from Hugging Face (**Qwen3-4B-Instruct**) acts as an analyst. It is **not** given the answers. It gets **tools** that query MongoDB and decides on its own which data to pull, and in what order. Then a **fact-checker** compares every number and every STRESSED/HEALTHY flag in its brief with the data the tools returned. If anything doesn't match, the agent has to revise. The finished brief is saved to MongoDB.

| Tool | What it returns |
|---|---|
| `get_national_overview` | model accuracy, coverage, the 7-of-8 scorecard, business-value numbers |
| `rank_states` | top/bottom major corn states by satellite corn health, corn share, or USDA yield change |
| `get_state_report` | one state: the satellite signal (Aug 2022) and USDA's final result (Jan 2023), kept separate |
| `get_market_reaction` | corn price moves on 2022 USDA report days, with direction (rose/fell) |

You'll see each step printed: the tool it chose, what it asked for, a preview of what came back, and the fact-check result.

**Tip:** Parts A–E only need to run once, since everything is stored in MongoDB. Next time, run the Setup cells and jump straight here.

In [ ]:
# ---------- Tools: plain Python functions that query MongoDB ----------
def r3(v):
    return round(v, 3) if isinstance(v, float) else v

def get_national_overview():
    m = db.model_metrics.find_one({}, {"_id": 0})
    sc = m.get("flag_scorecard", {})
    return {"satellite_model": {k: m.get(k) for k in ["backbone", "IoU_Corn", "mIoU", "acreage_correlation", "reference_ibm_corn_iou"]},
            "images_analyzed": db.chips.count_documents({}),
            "states_covered": db.states.count_documents({}),
            "satellite_flag_scorecard": {"flags_matching_usda": sc.get("correct"), "flags_total": sc.get("total")},
            "business_value": m.get("business_value"),
            "timing": "Satellite signals come from Jul-Aug 2022 photos (available in-season). USDA final yields were published in Jan 2023."}

def rank_states(metric: str = "corn_health", top_n: int = 5, lowest_first: bool = True):
    field = {"corn_health": "corn_health_peak", "yield_change": "yield_change_pct", "corn_share": "corn_share_pred"}.get(metric, "corn_health_peak")
    q = {"satellite_flag": {"$ne": "not a major corn state / too few photos"}, field: {"$ne": None}}
    rows = db.states.find(q, {"_id": 0}).sort(field, 1 if lowest_first else -1).limit(int(top_n))
    return {"ranked_by": metric, "order": "lowest first" if lowest_first else "highest first",
            "universe": "major corn states (>= 300 million bushels/yr on average in 2017-21)",
            "states": [{"state": r["state"].title(),
                        "satellite_corn_health_ndvi_aug_2022": r3(r["corn_health_peak"]),
                        "satellite_flag_aug_2022": r["satellite_flag"],
                        "usda_final_jan_2023": {
                            "yield_2022_bu_per_acre": r3(r.get("usda_yield_2022")),
                            "avg_yield_2017_21_bu_per_acre": r3(r.get("usda_yield_avg_17_21")),
                            "yield_change_vs_avg_pct": r3(r.get("yield_change_pct")),
                            "production_2022_million_bu": r3(r["production_2022_bu"] / 1e6) if r.get("production_2022_bu") else None}}
                       for r in rows]}

def get_state_report(state: str):
    s = db.states.find_one({"state": state.upper().strip()}, {"_id": 0})
    if not s:
        return {"error": f"No data for '{state}'. Use full US state names, e.g. 'Iowa'."}
    n_ranked = db.states.count_documents({"health_rank": {"$ne": None}})
    rank = f"{int(s['health_rank'])} of {n_ranked} states with photos (1 = healthiest)" if s.get("health_rank") is not None else None
    return {"state": s["state"].title(),
            "satellite_signal_available_aug_2022": {
                "corn_share_of_land_pct": r3(100 * s["corn_share_pred"]) if s.get("corn_share_pred") is not None else None,
                "corn_health_ndvi": r3(s.get("corn_health_peak")),
                "peak_season_photos": s.get("peak_photos"),
                "health_rank": rank,
                "flag": s.get("satellite_flag")},
            "usda_final_results_published_jan_2023": {
                "yield_2022_bu_per_acre": r3(s.get("usda_yield_2022")),
                "avg_yield_2017_21_bu_per_acre": r3(s.get("usda_yield_avg_17_21")),
                "yield_change_vs_avg_pct": r3(s.get("yield_change_pct")),
                "production_2022_million_bu": r3(s["production_2022_bu"] / 1e6) if s.get("production_2022_bu") else None}}

def get_market_reaction(month: str = "all"):
    all_rows = list(db.wasde_reports.find({}, {"_id": 0}).sort("date", 1))
    rows = all_rows
    if month and month.lower() != "all":
        rows = [r for r in all_rows if pd.Timestamp(r["date"]).strftime("%B").lower()[:3] in month.lower()] or all_rows
    v = db.model_metrics.find_one({}, {"business_value": 1})["business_value"]
    days = [{"date": r["date"], "close_usd_per_bushel": round(r["close_cents"] / 100, 2), "move_pct": r["move_pct"],
             "direction": "price ROSE" if r["move_pct"] > 0.05 else "price FELL" if r["move_pct"] < -0.05 else "price flat"}
            for r in rows]
    mv = [r["move_pct"] for r in all_rows]
    return {"wasde_report_days": days,
            "all_2022_report_days": f"price rose on {sum(m > 0.05 for m in mv)}, fell on {sum(m < -0.05 for m in mv)}, "
                                    f"flat on {sum(abs(m) <= 0.05 for m in mv)} of {len(mv)} report days",
            "avg_size_of_move_on_report_days_pct_ignoring_direction": v["avg_wasde_day_move_pct"],
            "avg_size_of_move_on_normal_days_pct_ignoring_direction": v["avg_normal_day_move_pct"],
            "biggest_report_day": {"date": v["biggest_wasde_day"], "move_pct": v["biggest_wasde_day_move_pct"],
                                   "usd_per_futures_contract": v["biggest_wasde_day_move_per_contract_usd"]},
            "coverage": "Only the 12 WASDE reports of calendar year 2022 are in the database. Nothing from 2023."}

def save_brief(title, audience, brief, unverified=None, steps=None):
    """Store a finished brief in MongoDB (called by the agent loop after the fact-check)."""
    doc = {"title": title, "audience": audience, "brief": brief, "model": LLM_ID,
           "unverified_numbers": unverified or [], "tool_steps": steps or [],
           "created_at": pd.Timestamp.now(tz="UTC").isoformat()}
    return {"saved": True, "id": str(db.briefs.insert_one(doc).inserted_id)}

TOOLS = {f.__name__: f for f in [get_national_overview, rank_states, get_state_report, get_market_reaction]}

def schema(name, desc, props=None, required=()):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props or {}, "required": list(required)}}}

TOOL_SCHEMAS = [
    schema("get_national_overview", "Satellite model accuracy, coverage, scorecard and business-value numbers (US corn crop value, price moves). Call this first."),
    schema("rank_states", "Rank the major US corn states by a metric.",
           {"metric": {"type": "string", "enum": ["corn_health", "yield_change", "corn_share"]},
            "top_n": {"type": "integer"}, "lowest_first": {"type": "boolean"}}),
    schema("get_state_report", "Satellite signal (Aug 2022) and USDA final result (Jan 2023) for one US state.",
           {"state": {"type": "string", "description": "Full state name, e.g. 'Kansas'"}}, ["state"]),
    schema("get_market_reaction", "Corn futures moves on 2022 USDA WASDE report days, with direction.",
           {"month": {"type": "string", "description": "Month name(s), or 'all'"}}),
]
print("Tools ready:", list(TOOLS))

In [ ]:
# ---------- Load the open LLM from Hugging Face ----------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_ID = "Qwen/Qwen3-4B-Instruct-2507"          # ~8 GB, fits the free T4 GPU
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                                           device_map="auto")
print("✅ Loaded", LLM_ID, "on", llm.device)

In [ ]:
# ---------- The agent loop: think → call tools → read results → write → fact-check → revise → save ----------
from IPython.display import Markdown, display

SYSTEM = (
    "You are CornSight, an agricultural market analyst. Your tools query a MongoDB database built from NASA/IBM Prithvi "
    "satellite analysis of US corn fields in 2022, USDA yield statistics and corn futures prices.\n"
    "Rules:\n"
    "1. Gather facts with tools before writing. Every number in your brief must appear in a tool result. "
    "Never invent numbers, dates or reports.\n"
    "2. Use rank_states to find the states that matter. Make at most 3 tool calls per turn.\n"
    "3. Timing: satellite signals were available in August 2022; USDA final yields were published in January 2023. "
    "The satellite measures crop HEALTH (NDVI); it does NOT predict a yield in bushels. Never say the satellite predicted a yield number.\n"
    "4. Call get_market_reaction before writing Market impact. For price moves, follow the 'direction' field: "
    "a positive move_pct means the price ROSE.\n"
    "5. Cause and effect: nobody outside this project saw our satellite results in 2022. Prices moved because of USDA "
    "reports and the market, never because of our satellite. Say whether the satellite signal AGREED or DISAGREED with what happened.\n"
    "6. Lower corn health (NDVI) and a worse health rank mean HIGHER crop risk. Only describe months and states you actually looked up. "
    "Copy each state's flag (STRESSED / HEALTHY / normal) exactly as the tool gives it.\n"
    "7. When ready, reply with the brief only, in Markdown: a first line '# <title>', then the sections "
    "## Summary, ## Satellite signals (Aug 2022), ## What USDA later confirmed (Jan 2023), ## Market impact, "
    "## Recommendation, ## Limitations. Do not write function calls in the brief."
)

def parse_tool_calls(text):
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", text, flags=re.S):
        try:
            c = json.loads(block.strip())
            calls.append({"name": c["name"], "arguments": c.get("arguments", {}) or {}})
        except Exception:
            pass
    return calls

def generate(messages, max_new_tokens=1200):
    prompt = tokenizer.apply_chat_template(messages, tools=TOOL_SCHEMAS, add_generation_prompt=True,
                                           tokenize=False, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---------- Fact-checker: every number in the brief must trace back to a tool result ----------
NUM_RE = re.compile(r"(?<![\w.])[-−]?\d[\d,]*(?:\.\d+)?")

def numbers_in(text):
    found = []
    for m in NUM_RE.finditer(text):
        raw = m.group().replace("−", "-").replace(",", "")
        try:
            v = float(raw)
        except ValueError:
            continue
        dec = len(raw.split(".")[1]) if "." in raw else 0
        if dec == 0 and (abs(v) <= 31 or 2000 <= abs(v) <= 2030):   # skip small counts, days, years
            continue
        found.append((m.group(), v, dec))
    return found

def fact_check(brief, tool_outputs):
    pool = set()
    for t in tool_outputs:
        for x in re.findall(r"-?\d+(?:\.\d+)?(?:e[+-]?\d+)?", t):
            f = abs(float(x))
            pool.update({f, f * 100, f / 100, f / 1e6, f / 1e9})        # allow %, cents→$, millions, billions
    def matches(p, v, dec):                                             # rounded (11.195→11.20) or cut off (11.195→11.19)
        return round(p, dec) == round(v, dec) or 0 <= p - v < 10 ** -dec
    return sorted({raw for raw, v, dec in numbers_in(brief) if not any(matches(p, abs(v), dec) for p in pool)})

# ---------- Flag checker: every STRESSED / HEALTHY claim must match the tool results ----------
def collect_flags(result, flags):
    if isinstance(result, dict):
        sig = result.get("satellite_signal_available_aug_2022")
        fl = result.get("satellite_flag_aug_2022") or (sig.get("flag") if isinstance(sig, dict) else None)
        if result.get("state") and fl:
            flags[result["state"]] = fl.split()[0].upper()              # STRESSED / HEALTHY / NORMAL
        for v in result.values():
            collect_flags(v, flags)
    elif isinstance(result, list):
        for v in result:
            collect_flags(v, flags)

def flag_check(brief, flags, looked_up):
    problems = set()
    for sent in re.split(r"(?<=[.!?])\s+|\n", brief):
        words = [w for w in ("STRESSED", "HEALTHY") if w in sent]
        if len(words) != 1:                                             # only sentences making one flag claim
            continue
        named = [s for s in flags if re.search(r"\b" + re.escape(s) + r"\b", sent, re.I)]
        if re.search(r"\ball (three|of them|these states|the states)\b", sent, re.I):
            named += list(looked_up)
        problems |= {f"{s} is {flags[s]}, not {words[0]}" for s in named if flags.get(s) != words[0]}
    return sorted(problems)

def clean_reply(text):
    text = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S)
    return re.split(r"\n\s*`*save_brief\s*\(", text)[0].strip()     # drop any function call the model typed out

MAX_CALLS_PER_STEP = 4

def run_agent(task, audience="analyst", max_steps=12, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
    trace, tool_outputs, revised, nudged = [], [], False, False
    flags, looked_up = {}, []
    for step in range(1, max_steps + 1):
        reply = generate(messages)
        calls = parse_tool_calls(reply)
        if not calls and not trace:                     # guardrail: no answering from memory
            print("⚠️  Agent tried to answer without using any tools → told to gather data first")
            messages.append({"role": "assistant", "content": clean_reply(reply)})
            messages.append({"role": "user", "content": "You have not used any tools yet. Call get_national_overview "
                             "and the other tools to gather data first, then write the brief."})
            continue
        if not calls and not nudged and "get_market_reaction" not in {t["tool"] for t in trace}:
            nudged = True                               # guardrail: the Market impact section needs real price data
            print("⚠️  Agent skipped the price data → told to call get_market_reaction first")
            messages.append({"role": "assistant", "content": clean_reply(reply)})
            messages.append({"role": "user", "content": "Before finishing, call get_market_reaction to get the 2022 "
                             "price data for the Market impact section, then rewrite the full brief."})
            continue
        if not calls:                                   # no tool calls → the agent wrote its brief
            brief = clean_reply(reply)
            bad = fact_check(brief, tool_outputs)
            wrong_flags = flag_check(brief, flags, looked_up)
            if (bad or wrong_flags) and not revised:    # self-correction: one revision round
                revised = True
                problems = []
                if bad:
                    print(f"🔎 Fact-check: {len(bad)} number(s) not found in any tool result: {', '.join(bad)} → agent must revise")
                    problems.append("These numbers do not appear in any tool result: " + ", ".join(bad) +
                                    ". For each one, either call a tool to get the real value, or delete that number.")
                if wrong_flags:
                    print(f"🔎 Flag check: {'; '.join(wrong_flags)} → agent must revise")
                    problems.append("These satellite flags are wrong: " + "; ".join(wrong_flags) +
                                    ". Use each state's flag exactly as the tool gives it.")
                messages.append({"role": "assistant", "content": brief})
                messages.append({"role": "user", "content": "Fact-check failed. " + " ".join(problems) +
                                 " Then rewrite the full brief. Keep the same format."})
                continue
            if not bad and not wrong_flags:
                print("🔎 Fact-check: every number and every satellite flag matches the tool results ✓")
            else:
                print(f"🔎 Fact-check after revision: still wrong → {', '.join(bad + wrong_flags)} (flagged for human review)")
            title = next((l.lstrip("# ").strip() for l in brief.splitlines() if l.strip()), "CornSight brief")
            saved = save_brief(title[:150], audience, brief, unverified=bad + wrong_flags, steps=trace)
            print(f"💾 Saved to MongoDB → briefs collection (id {saved['id']})")
            return brief, trace
        if len(calls) > MAX_CALLS_PER_STEP:             # guardrail: keep the agent focused
            print(f"⚠️  Step {step}: agent asked for {len(calls)} tools at once; running the first {MAX_CALLS_PER_STEP}")
            calls = calls[:MAX_CALLS_PER_STEP]
        messages.append({"role": "assistant", "content": re.sub(r"<tool_call>.*?</tool_call>", "", reply, flags=re.S).strip(),
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            fn = TOOLS.get(c["name"])
            try:
                result = fn(**c["arguments"]) if fn else {"error": f"unknown tool {c['name']}"}
            except Exception as e:
                result = {"error": str(e)}
            out = json.dumps(result, default=str)
            tool_outputs.append(out)
            collect_flags(result, flags)
            if c["name"] == "get_state_report" and isinstance(result, dict) and "state" in result:
                looked_up.append(result["state"])
            trace.append({"step": step, "tool": c["name"], "args": c["arguments"]})
            if verbose:
                args = ", ".join(f"{k}={v!r}" for k, v in c["arguments"].items())
                print(f"🔧 Step {step}: {c['name']}({args})\n   ↳ {out[:220]}{'…' if len(out) > 220 else ''}")
            messages.append({"role": "tool", "name": c["name"], "content": out})
    return "(agent stopped: too many steps)", trace

### Task 1 — Brief for a grain trading desk

In [ ]:
TASK_TRADER = ("You work for a grain trading desk. It is late August 2022. Using the satellite signals, identify which "
               "major corn states look stressed or healthy, explain what that implies for the US harvest and corn prices, "
               "and then check against what USDA later confirmed. Audience: commodity traders.")
brief_trader, trace_trader = run_agent(TASK_TRADER, audience="commodity traders")
display(Markdown(brief_trader))

### Task 2 — Risk check for an agricultural lender

In [ ]:
TASK_LENDER = ("You advise an agricultural bank with farm-loan exposure in Kansas, Nebraska and Iowa. Using the satellite "
               "signals for each of these states, assess 2022 crop risk for borrowers in each, rank the three by risk, and "
               "recommend actions (e.g. extra monitoring, loan reviews). Then compare with USDA's final yields. "
               "Audience: credit risk committee.")
brief_lender, trace_lender = run_agent(TASK_LENDER, audience="credit risk committee")
display(Markdown(brief_lender))

In [ ]:
# Everything the agent wrote is stored in MongoDB
print(f"Briefs stored in MongoDB: {db.briefs.count_documents({})}")
for b in db.briefs.find({}, {"_id": 0, "title": 1, "audience": 1, "created_at": 1, "unverified_numbers": 1}) \
                  .sort("created_at", -1).limit(6):
    u = b.get("unverified_numbers")
    check = "not fact-checked (older run)" if u is None else "✓ verified" if not u else f"⚠ unverified: {u}"
    print(f"  • {b['created_at'][:19]}  [{b['audience']}]  {b['title']}  ({check})")

# Also save the two briefs as Markdown files (Colab Files panel → cornsight_outputs/) for the slides
for name, text in [("brief_trader.md", brief_trader), ("brief_lender.md", brief_lender)]:
    with open(os.path.join(OUT_DIR, name), "w", encoding="utf-8") as f:
        f.write(text)
print("Saved brief_trader.md and brief_lender.md to", OUT_DIR)

## ✅ Done
The briefs are stored in MongoDB (`briefs` collection) with the tools the agent used and the fact-check result.
If a cell shows a red error, send back the Part and the full message.

Everything is stored in MongoDB. Copies of the new charts and both briefs are in the Colab **Files** panel → `cornsight_outputs/`.